In [1]:
import os
import numpy as np
import pandas as pd

# Resolve project root robustly (works from repo root or Notebooks_Ours/preprocessing)
cwd = os.getcwd()
cwd_base = os.path.basename(cwd)
if cwd_base == "preprocessing":
    PROJECT_ROOT = os.path.abspath(os.path.join(cwd, "..", ".."))
elif cwd_base == "Notebooks_Ours":
    PROJECT_ROOT = os.path.abspath(os.path.join(cwd, ".."))
else:
    PROJECT_ROOT = os.path.abspath(cwd)

PROVIDED_DIR = os.path.join(PROJECT_ROOT, "Datasets_Provided")
NEW_DIR = os.path.join(PROJECT_ROOT, "Datasets_Ours")
OUTPUT_DIR = NEW_DIR

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Expected training files
EXPECTED_FILES = {
    "gaia": "gaia_features_training.csv",
    "jrc_gsw": "jrc_gsw_features_training.csv",
    "landsat_allbands": "landsat_features_training_allbands.csv",
    "terraclimate_allvars": "terraclimate_features_training_allvars.csv",
    "esa_cci": "esa_cci_features_training.csv",
    "raster_buffer": "esa_jrc_gaia_buffer_training.csv",
}

def resolve_path(fname: str) -> str | None:
    """Return the first existing path for fname across Provided and New dirs."""
    for base in (PROVIDED_DIR, NEW_DIR):
        p = os.path.join(base, fname)
        if os.path.exists(p):
            return p
    return None

resolved = {}
missing = []
for key, fname in EXPECTED_FILES.items():
    p = resolve_path(fname)
    if p is None:
        missing.append((key, fname))
    else:
        resolved[key] = p

print("Resolved input paths:")
for k, p in resolved.items():
    print(f" - {k:18s} -> {os.path.abspath(p)}")

if missing:
    msg = "Missing these expected files in BOTH folders:\n" + "\n".join([f"{k}: {f}" for k, f in missing])
    raise FileNotFoundError(msg)

print("\nAll expected files found")

In [3]:
def standardize_join_keys(df: pd.DataFrame) -> pd.DataFrame:
    """Standardize join columns to: latitude, longitude, sample_date."""
    out = df.copy()
    rename_map = {}
    for c in out.columns:
        lc = c.strip().lower()
        if lc in {"latitude", "lat"} or lc.startswith("lat"):
            rename_map[c] = "latitude"
        elif lc in {"longitude", "lon", "lng"} or lc.startswith("lon"):
            rename_map[c] = "longitude"
        # Be conservative: only map obvious sample-date columns
        elif lc in {"sample date", "sample_date"}:
            rename_map[c] = "sample_date"
    out = out.rename(columns=rename_map)

    # If a dataset used a generic "Date" column, map it ONLY if sample_date doesn't exist yet
    if "sample_date" not in out.columns:
        for c in list(out.columns):
            if c.strip().lower() == "date":
                out = out.rename(columns={c: "sample_date"})
                break

    # Drop duplicate columns created by renaming collisions
    out = out.loc[:, ~out.columns.duplicated(keep="first")]

    required = {"latitude", "longitude", "sample_date"}
    missing = required - set(out.columns)
    if missing:
        raise ValueError(f"Missing required join columns after standardization: {missing}")

    # helper parsed date (not used for join)
    out["sample_date_parsed"] = pd.to_datetime(out["sample_date"], errors="coerce")
    return out

In [4]:
# Load + standardize
datasets = {}
for name, path in resolved.items():
    df = pd.read_csv(path)
    df_std = standardize_join_keys(df)
    datasets[name] = df_std
    print(f"{name:18s} shape={df_std.shape}  cols={len(df_std.columns)}")

gaia               shape=(9319, 9)  cols=9
jrc_gsw            shape=(9319, 13)  cols=13
landsat_allbands   shape=(9319, 16)  cols=16
terraclimate_allvars shape=(9319, 18)  cols=18
esa_cci            shape=(9319, 9)  cols=9
raster_buffer      shape=(9319, 26)  cols=26


In [5]:
# Proper merge on join keys + output Final Datasets
merge_keys = ["latitude", "longitude", "sample_date"]

VALIDATION_FILES = {
    "gaia": "gaia_features_validation.csv",
    "jrc_gsw": "jrc_gsw_features_validation.csv",
    "landsat_allbands": "landsat_features_validation_allbands.csv",
    "terraclimate_allvars": "terraclimate_features_validation_allvars.csv",
    "esa_cci": "esa_cci_features_validation.csv",
    "raster_buffer": "esa_jrc_gaia_buffer_validation.csv",
}

resolved_val = {}
missing_val = []
for key, fname in VALIDATION_FILES.items():
    p = resolve_path(fname)
    if p is None:
        missing_val.append((key, fname))
    else:
        resolved_val[key] = p

if missing_val:
    print("Missing validation files:")
    for k, f in missing_val:
        print(f" - {k}: {f}")
    raise FileNotFoundError("Missing one or more validation feature files")

# Load + standardize validation datasets
val_datasets = {}
for name, path in resolved_val.items():
    df = pd.read_csv(path)
    df_std = standardize_join_keys(df)
    val_datasets[name] = df_std
    print(f"VAL {name:18s} shape={df_std.shape}  cols={len(df_std.columns)}")


def merge_feature_datasets(datasets_dict):
    merged_out = None
    for name, df in datasets_dict.items():
        if merged_out is None:
            merged_out = df
        else:
            merged_out = pd.merge(
                merged_out,
                df,
                on=merge_keys,
                how="outer",
                suffixes=("", f"__{name}")
            )

    cols_to_drop = [
        c for c in merged_out.columns
        if ("date" in c.lower()) and (c.lower() != "sample_date")
    ]
    if cols_to_drop:
        print("Dropping from dataframe:")
        print(cols_to_drop)
        merged_out = merged_out.drop(columns=cols_to_drop)
    return merged_out

# Merge training + validation feature datasets
merged_train = merge_feature_datasets(datasets)
merged_val = merge_feature_datasets(val_datasets)

# Keep legacy name for later cells
merged = merged_train

print("Final training feature shape:", merged_train.shape)
print("Final validation feature shape:", merged_val.shape)

# --- Proper join with targets (3-key) ---
wq_data = pd.read_csv(os.path.join(PROJECT_ROOT, "Datasets_Provided", "water_quality_training_dataset.csv"))
wq_data.columns = wq_data.columns.str.lower().str.replace(" ", "_")

merged_train["sample_date"] = pd.to_datetime(merged_train["sample_date"], format="%d-%m-%Y", errors="coerce")
wq_data["sample_date"] = pd.to_datetime(wq_data["sample_date"], format="%d-%m-%Y", errors="coerce")

merged_train["latitude"] = merged_train["latitude"].round(6)
merged_train["longitude"] = merged_train["longitude"].round(6)
wq_data["latitude"] = wq_data["latitude"].round(6)
wq_data["longitude"] = wq_data["longitude"].round(6)

# Handle censored DRP
DRP_LOD_10 = 10.0
DRP_LOD_20 = 20.0

def prepare_drp_for_ml(df, drp_col="dissolved_reactive_phosphorus", random_state=42):
    out = df.copy()
    rng = np.random.default_rng(random_state)
    drp = out[drp_col].astype(float)
    out["drp_censored_10"] = (drp == DRP_LOD_10).astype(int)
    out["drp_censored_20"] = (drp == DRP_LOD_20).astype(int)
    mask_10 = (drp == DRP_LOD_10)
    mask_20 = (drp == DRP_LOD_20)
    out.loc[mask_10, drp_col] = rng.uniform(0, DRP_LOD_10, size=mask_10.sum())
    out.loc[mask_20, drp_col] = rng.uniform(DRP_LOD_10, DRP_LOD_20, size=mask_20.sum())
    return out

wq_data = prepare_drp_for_ml(wq_data)

wq_targets = wq_data[[
    "latitude", "longitude", "sample_date",
    "total_alkalinity", "electrical_conductance", "dissolved_reactive_phosphorus",
    "drp_censored_10", "drp_censored_20",
]].copy()

complete_data = merged_train.merge(
    wq_targets,
    on=["latitude", "longitude", "sample_date"],
    how="inner"
)

# Feature list (exclude join keys)
feature_cols = [c for c in merged_train.columns if c not in ["latitude", "longitude", "sample_date"]]

# Ensure validation has the same feature columns
missing_features = [c for c in feature_cols if c not in merged_val.columns]
for c in missing_features:
    merged_val[c] = np.nan

# month_fitted
from pathlib import Path
import pickle
params_path = Path(PROJECT_ROOT) / "Notebooks_Ours" / "preprocessing" / "fitted_monthly_params.pkl"
with open(params_path, "rb") as f:
    fitted_params = pickle.load(f)

complete_data["month"] = pd.to_datetime(complete_data["sample_date"]).dt.month
merged_val["month"] = pd.to_datetime(merged_val["sample_date"], format="%d-%m-%Y", errors="coerce").dt.month

# DRP
params = fitted_params["DRP"]["params"]
complete_data["month_fitted_drp"] = (
    params[0] * complete_data["month"]**3 +
    params[1] * complete_data["month"]**2 +
    params[2] * complete_data["month"] +
    params[3]
)
merged_val["month_fitted_drp"] = (
    params[0] * merged_val["month"]**3 +
    params[1] * merged_val["month"]**2 +
    params[2] * merged_val["month"] +
    params[3]
)

# EC
params = fitted_params["EC"]["params"]
complete_data["month_fitted_ec"] = (
    params[0] * np.sin(params[1] * complete_data["month"] + params[2]) + params[3]
)
merged_val["month_fitted_ec"] = (
    params[0] * np.sin(params[1] * merged_val["month"] + params[2]) + params[3]
)

# TA
params = fitted_params["TA"]["params"]
complete_data["month_fitted_ta"] = (
    params[0] * np.sin(params[1] * complete_data["month"] + params[2]) + params[3]
)
merged_val["month_fitted_ta"] = (
    params[0] * np.sin(params[1] * merged_val["month"] + params[2]) + params[3]
)

# Create datasets
output_path = Path(PROJECT_ROOT) / "Datasets_Ours" / "Final Datasets"
output_path.mkdir(parents=True, exist_ok=True)

# Training
train_base = complete_data[["latitude", "longitude"] + feature_cols].copy()

# DRP
train_drp = train_base.copy()
train_drp["dissolved_reactive_phosphorus"] = complete_data["dissolved_reactive_phosphorus"]
train_drp["drp_censored_10"] = complete_data["drp_censored_10"]
train_drp["drp_censored_20"] = complete_data["drp_censored_20"]
train_drp["month_fitted"] = complete_data["month_fitted_drp"]

# EC
train_ec = train_base.copy()
train_ec["electrical_conductance"] = complete_data["electrical_conductance"]
train_ec["month_fitted"] = complete_data["month_fitted_ec"]

# TA
train_ta = train_base.copy()
train_ta["total_alkalinity"] = complete_data["total_alkalinity"]
train_ta["month_fitted"] = complete_data["month_fitted_ta"]

# Validation
val_base = merged_val[["latitude", "longitude"] + feature_cols].copy()

val_drp = val_base.copy()
val_drp["month_fitted"] = merged_val["month_fitted_drp"]

val_ec = val_base.copy()
val_ec["month_fitted"] = merged_val["month_fitted_ec"]

val_ta = val_base.copy()
val_ta["month_fitted"] = merged_val["month_fitted_ta"]

# Save
train_drp.to_csv(output_path / "drp_training_complete.csv", index=False)
train_ec.to_csv(output_path / "ec_training_complete.csv", index=False)
train_ta.to_csv(output_path / "ta_training_complete.csv", index=False)

val_drp.to_csv(output_path / "drp_validation.csv", index=False)
val_ec.to_csv(output_path / "ec_validation.csv", index=False)
val_ta.to_csv(output_path / "ta_validation.csv", index=False)


VAL gaia               shape=(200, 9)  cols=9
VAL jrc_gsw            shape=(200, 13)  cols=13
VAL landsat_allbands   shape=(200, 16)  cols=16
VAL terraclimate_allvars shape=(200, 18)  cols=18
VAL esa_cci            shape=(200, 9)  cols=9
VAL raster_buffer      shape=(200, 26)  cols=26
Dropping from dataframe:
['sample_date_parsed', 'sample_date_parsed__jrc_gsw', 'sample_date_parsed__landsat_allbands', 'sample_date_parsed__terraclimate_allvars', 'sample_date_parsed__esa_cci', 'sample_date_parsed__raster_buffer']
Dropping from dataframe:
['sample_date_parsed', 'sample_date_parsed__jrc_gsw', 'sample_date_parsed__landsat_allbands', 'sample_date_parsed__terraclimate_allvars', 'sample_date_parsed__esa_cci', 'sample_date_parsed__raster_buffer']
Final training feature shape: (9319, 70)
Final validation feature shape: (200, 70)


In [6]:
# shape checking (Final Datasets)
from pathlib import Path

final_dir = Path(PROJECT_ROOT) / "Datasets_Ours" / "Final Datasets"

train_drp = pd.read_csv(final_dir / "drp_training_complete.csv")
train_ec = pd.read_csv(final_dir / "ec_training_complete.csv")
train_ta = pd.read_csv(final_dir / "ta_training_complete.csv")

val_drp = pd.read_csv(final_dir / "drp_validation.csv")
val_ec = pd.read_csv(final_dir / "ec_validation.csv")
val_ta = pd.read_csv(final_dir / "ta_validation.csv")

print("Training shapes:")
print("  DRP:", train_drp.shape)
print("  EC :", train_ec.shape)
print("  TA :", train_ta.shape)

print("\nValidation shapes:")
print("  DRP:", val_drp.shape)
print("  EC :", val_ec.shape)
print("  TA :", val_ta.shape)

print("\nFeature column counts (train):")
print("  DRP:", train_drp.drop(columns=["dissolved_reactive_phosphorus", "drp_censored_10", "drp_censored_20"]).shape[1])
print("  EC :", train_ec.drop(columns=["electrical_conductance"]).shape[1])
print("  TA :", train_ta.drop(columns=["total_alkalinity"]).shape[1])

print("\nFeature column counts (val):")
print("  DRP:", val_drp.shape[1])
print("  EC :", val_ec.shape[1])
print("  TA :", val_ta.shape[1])


Training shapes:
  DRP: (9319, 73)
  EC : (9319, 71)
  TA : (9319, 71)

Validation shapes:
  DRP: (200, 70)
  EC : (200, 70)
  TA : (200, 70)

Feature column counts (train):
  DRP: 70
  EC : 70
  TA : 70

Feature column counts (val):
  DRP: 70
  EC : 70
  TA : 70
